### Torch Archiver & Torch Serve

In [104]:
import torch
import torch.nn as nn

model = nn.Linear(4,1)
model.eval()
script_module = torch.jit.script(model)
model_file = f"model.pt"
script_module.save(model_file)

In [133]:
model(torch.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [1.0, 2.0, 3.0, 5.0]
]))

tensor([[2.3237],
        [2.5380]], grad_fn=<AddmmBackward0>)

In [149]:
%%writefile handler.py
import torch
from ts.torch_handler.base_handler import BaseHandler

class Handler(BaseHandler):
    def initialize(self, context):
        model_dir = context.system_properties.get("model_dir")
        model_path = f"{model_dir}/model.pt"
        self.model = torch.jit.load(model_path)
        self.model.eval()

    def preprocess(self, data):
        row = data[0]
        body = row.get("body", {})
        values = body.get("data", [])
        tensor = torch.tensor(
            values,
            dtype=torch.float32,
        )
        if tensor.dim() == 1:
            tensor = tensor.unsqueeze(0)
        return tensor
    
    
    def inference(self, inputs):
        with torch.no_grad():
            outputs = self.model(inputs)
        return outputs
    
    def postprocess(self, outputs):
        return [outputs.tolist()]

Overwriting handler.py


In [161]:
!ls | grep "model.mar"

model.mar


In [150]:
!torch-model-archiver -f \
  --model-name "model" \
  --version 1.0 \
  --serialized-file 'model.pt' \
  --handler 'handler.py' \
  --export-path '.'

WARNING - Overwriting ./model.mar ...


In [160]:
!ls | grep "model.mar"

model.mar


In [111]:
!pwd

/home/ridwanfatur/work/learning/youtube-channel/notebooks


In [99]:
!torchserve --start \
  --model-store . \
  --models model=model.mar \
  --ncs \
  --disable-token-auth

TorchServe is already running, please use torchserve --stop to stop TorchServe.


In [157]:
!cat logs/model_log.log

2026-08-27T11:19:50,503 [INFO ] W-9000-model_1.0-stdout MODEL_LOG - s_name_part0=/tmp/.ts.sock, s_name_part1=9000, pid=39310
2026-08-27T11:19:50,504 [INFO ] W-9000-model_1.0-stdout MODEL_LOG - Listening on port: /tmp/.ts.sock.9000
2026-08-27T11:19:50,509 [INFO ] W-9000-model_1.0-stdout MODEL_LOG - Successfully loaded /home/ridwanfatur/miniconda3/envs/py3_12_9/lib/python3.12/site-packages/ts/configs/metrics.yaml.
2026-08-27T11:19:50,510 [INFO ] W-9000-model_1.0-stdout MODEL_LOG - [PID]39310
2026-08-27T11:19:50,511 [INFO ] W-9000-model_1.0-stdout MODEL_LOG - Torch worker started.
2026-08-27T11:19:50,511 [INFO ] W-9000-model_1.0-stdout MODEL_LOG - Python runtime: 3.12.9
2026-08-27T11:19:50,520 [INFO ] W-9000-model_1.0-stdout MODEL_LOG - Connection accepted: /tmp/.ts.sock.9000.
2026-08-27T11:19:50,553 [INFO ] W-9000-model_1.0-stdout MODEL_LOG - model_name: model, batchSize: 1
2026-08-27T11:19:50,801 [INFO ] W-9000-model_1.0-stdout MODEL_LOG - OpenVINO is not enabled
2026-08-27T11:19:50,820

In [158]:
!torchserve --stop

TorchServe has stopped.


In [152]:
!curl http://localhost:8081/models

{
  "models": [
    {
      "modelName": "model",
      "modelUrl": "model.mar"
    }
  ]
}


In [154]:
!curl -X POST http://localhost:8080/predictions/model \
  -H "Content-Type: application/json" \
  -d '{"data": [[1.0, 2.0, 3.0, 4.0],[1.0, 2.0, 3.0, 9.0]]}'

[
  [
    2.3237223625183105
  ],
  [
    3.3951992988586426
  ]
]

### Docker Image

In [159]:
# serving_container_uri = "us-docker.pkg.dev/vertex-ai/prediction/pytorch-cpu.1-11:latest"

In [162]:
!ls | grep "model.mar"

model.mar


In [163]:
!mkdir model_serve

In [164]:
!cp ./model.mar ./model_serve/

In [166]:
!ls model_serve | grep "model.mar"

model.mar


In [167]:
!docker run -d \
    --name simple-model \
    -p 8080:8080 \
    -v $(pwd)/model_serve:/tmp/model \
    -e AIP_STORAGE_URI=/tmp/model \
    -e AIP_HTTP_PORT=8080 \
    -e AIP_HEALTH_ROUTE=/ping \
    -e AIP_PREDICT_ROUTE=/predictions/model \
    us-docker.pkg.dev/vertex-ai/prediction/pytorch-cpu.1-11:latest

9369f322ac5b4f3647af9a15fb45df7143fa38af07a2f06cf0da1d67d3601a6a


In [169]:
!docker ps

CONTAINER ID   IMAGE                                                            COMMAND                  CREATED          STATUS         PORTS                                                  NAMES
9369f322ac5b   us-docker.pkg.dev/vertex-ai/prediction/pytorch-cpu.1-11:latest   "python entrypoint.py"   10 seconds ago   Up 9 seconds   7070-7071/tcp, 8081-8082/tcp, 0.0.0.0:8080->8080/tcp   simple-model


In [176]:
# !docker rm -f simple-model

In [171]:
!curl http://localhost:8080/ping

{
  "status": "Healthy"
}


In [172]:
import requests

url = "http://localhost:8080/predictions/model"

payload = {
    "instances": [
        {
            "body": {
                "data": [
                    [1.0, 2.0, 3.0, 4.0],
                    [1.0, 2.0, 3.0, 9.0]
                ]
            }
        }        
    ]
}

response = requests.post(
    url,
    json=payload,
    headers={"Content-Type": "application/json"}
)

In [173]:
response.status_code

200

In [174]:
result = response.json()
result

{'predictions': [[[2.3237223625183105], [3.3951992988586426]]]}